# 02 — Dialect Identification Training (Local)
Fine-tunes PhoWhisper-base for 63-province Vietnamese dialect classification.

**Model**: `vinai/PhoWhisper-base`  
**Classes**: 63 provinces + 3 regions + 3 ethnicities  
**Dataset**: ViMD (102h) + ViMD-augmented  

> ⚠️ **Lưu ý**: Máy không có GPU — PhoWhisper-base trên CPU rất chậm.
> Cân nhắc dùng `vinai/PhoWhisper-base` thay thế cho việc phát triển local.


In [1]:
# ── Cell 0: Cấu hình đường dẫn local ──────────────────────────────────────────
import sys, os
from pathlib import Path

_nb_dir = Path(os.path.abspath('.')).resolve()
if _nb_dir.name == 'notebooks':
    PROJECT_ROOT = str(_nb_dir.parent.parent)
elif _nb_dir.name == 'training':
    PROJECT_ROOT = str(_nb_dir.parent)
else:
    PROJECT_ROOT = str(_nb_dir)
# PROJECT_ROOT = r'd:\FPTU\Semester 7\DAT301m\Project'

DATASET_PATH   = os.path.join(PROJECT_ROOT, 'storage', 'datasets', 'hf_dataset', 'vietspeech-vimd')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'storage', 'checkpoints', 'dialect')
EXPORT_DIR     = os.path.join(PROJECT_ROOT, 'storage', 'weights', 'dialect')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

import torch
try:
    torch.multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  CPU-only — huấn luyện sẽ rất chậm.')
    print('   Cân nhắc thay MODEL_BACKBONE = "microsoft/wavlm-base" (nhẹ hơn nhiều)')

print(f'\nPROJECT_ROOT  : {PROJECT_ROOT}')
print(f'DATASET_PATH  : {DATASET_PATH}')
print(f'CHECKPOINT_DIR: {CHECKPOINT_DIR}')


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB

PROJECT_ROOT  : /mnt/d/FPTU/Semester 7/DAT301m/Project
DATASET_PATH  : /mnt/d/FPTU/Semester 7/DAT301m/Project/storage/datasets/hf_dataset/vietspeech-vimd
CHECKPOINT_DIR: /mnt/d/FPTU/Semester 7/DAT301m/Project/storage/checkpoints/dialect


In [2]:
# ── Cell 1: Cài dependencies ────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'accelerate', 'evaluate', 'soundfile', 'mlflow'],
               check=False)
print('Dependencies installed')


Dependencies installed


In [3]:
# ── Cell 2: Load config ─────────────────────────────────────────────────────────
import yaml

config_path = os.path.join(PROJECT_ROOT, 'training', 'configs', 'dialect_config.yaml')
if os.path.exists(config_path):
    with open(config_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    print('Config loaded:', yaml.dump(cfg, default_flow_style=False))
else:
    print(f'⚠️  Config không tìm thấy tại {config_path}. Dùng defaults.')
    cfg = {
        'model': {'backbone': 'vinai/PhoWhisper-base'},  # base thay vì large cho CPU
        'training': {'batch_size': 4, 'num_epochs': 10, 'learning_rate': 1e-4,
                     'gradient_accumulation_steps': 4, 'warmup_steps': 100},
        'data': {'max_duration_s': 10.0},
    }
    print('Dùng config mặc định:', cfg)


Config loaded: checkpointing:
  gdrive_path: /content/drive/MyDrive/vietspeech_ai/checkpoints/dialect
  gdrive_sync: true
  keep_top_k: 3
  monitor_metric: province_accuracy
  monitor_mode: max
  save_every_n_steps: 500
data:
  augment_prob: 0.5
  augment_train: true
  freq_mask_param: 27
  max_duration_s: 20.0
  min_duration_s: 0.5
  num_freq_masks: 2
  num_time_masks: 2
  sample_rate: 16000
  spec_augment: true
  time_mask_param: 100
evaluation:
  metrics:
  - province_accuracy
  - region_accuracy
  - top3_accuracy
  - f1_macro
  - f1_weighted
  - confusion_matrix
  target_province_accuracy: 0.7
  target_region_accuracy: 0.95
logging:
  log_confusion_matrix: true
  log_every_n_steps: 50
  mlflow_experiment: dialect_identification
loss:
  arcface:
    weight: 0.5
  ethnic:
    type: cross_entropy
    weight: 0.2
  province:
    gamma: 2.0
    type: focal
    weight: 1.0
  region:
    type: cross_entropy
    weight: 0.3
model:
  adapter_size: 64
  arcface_margin: 0.3
  arcface_scale: 6

In [4]:
# ── Cell 3: Load HuggingFace Dataset ────────────────────────────────────────────
from datasets import load_from_disk, Audio

if not os.path.exists(DATASET_PATH):
    raise RuntimeError(
        f'Dataset không tìm thấy tại: {DATASET_PATH}\n'
        'Hãy chạy 01_dataset_prep.ipynb trước.'
    )

ds = load_from_disk(DATASET_PATH)
print('Dataset loaded:')
print(ds)

# ── Khôi phục raw audio từ file Parquet gốc ─────────────────────────────
# Do 01_dataset_prep đã drop cột audio, ta cần trích xuất lại để WavLM dùng
AUDIO_DIR = os.path.join(DATASET_PATH, 'audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

needed_filenames = set(ds['train']['filename']) | set(ds['valid']['filename']) | set(ds['test']['filename'])
extracted_files = set(os.listdir(AUDIO_DIR))
missing_files = needed_filenames - extracted_files

if missing_files:
    print(f'Đang giải nén {len(missing_files)} file audio từ parquet gốc (chạy lần đầu)...')
    import glob, pandas as pd
    from tqdm.auto import tqdm
    parquet_files = glob.glob(os.path.join(PROJECT_ROOT, 'ViMD', 'data', '**', '*.parquet'), recursive=True)
    for fpath in tqdm(parquet_files, desc='Trích xuất audio'):
        if not missing_files: break
        try:
            df = pd.read_parquet(fpath, columns=['filename', 'audio'])
            for _, row in df.iterrows():
                fn = row['filename']
                if fn in missing_files:
                    audio_field = row['audio']
                    b = None
                    if isinstance(audio_field, dict):
                        b = audio_field.get('bytes')
                    elif isinstance(audio_field, bytes):
                        b = audio_field
                    
                    if b:
                        with open(os.path.join(AUDIO_DIR, fn), 'wb') as audio_file:
                            audio_file.write(b)
                        missing_files.remove(fn)
        except Exception:
            pass

# Thêm đường dẫn audio vào Dataset
def add_audio_path(example):
    example['audio'] = os.path.join(AUDIO_DIR, example['filename'])
    return example

if 'audio' not in ds['train'].column_names:
    print('Thêm cột audio (đường dẫn file) vào dataset...')
    ds = ds.map(add_audio_path, desc='Mapping audio paths')

# ds = ds.cast_column('audio', Audio(sampling_rate=16000)) # Removed to avoid torchcodec bug

# Build label encoders
# ⚠️ FIX: dùng key 'valid' (không phải 'val') — theo DatasetSplitter
province_list   = sorted(ds['train'].unique('province'))
region_list     = ['North', 'Central', 'South']
ethnicity_list  = ['viet', 'ede', 'bahnar']

province2id  = {p: i for i, p in enumerate(province_list)}
region2id    = {r: i for i, r in enumerate(region_list)}
ethnicity2id = {e: i for i, e in enumerate(ethnicity_list)}

print(f'Provinces: {len(province2id)} | Regions: {len(region2id)} | Ethnicities: {len(ethnicity2id)}')


/home/justeg4n/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded:
DatasetDict({
    train: Dataset({
        features: ['duration', 'mel_mean', 'mel_std', 'energy_mean', 'energy_std', 'energy_max', 'pitch_mean', 'pitch_std', 'pitch_min', 'pitch_max', 'voiced_ratio', 'zcr_mean', 'zcr_std', 'spectral_centroid_mean', 'speaker_embedding', 'filename', 'text', 'speakerID', 'gender', 'province', 'region', 'ethnicity', 'language', 'is_augmented', 'aug_id'],
        num_rows: 28814
    })
    valid: Dataset({
        features: ['duration', 'mel_mean', 'mel_std', 'energy_mean', 'energy_std', 'energy_max', 'pitch_mean', 'pitch_std', 'pitch_min', 'pitch_max', 'voiced_ratio', 'zcr_mean', 'zcr_std', 'spectral_centroid_mean', 'speaker_embedding', 'filename', 'text', 'speakerID', 'gender', 'province', 'region', 'ethnicity', 'language', 'is_augmented', 'aug_id'],
        num_rows: 3604
    })
    test: Dataset({
        features: ['duration', 'mel_mean', 'mel_std', 'energy_mean', 'energy_std', 'energy_max', 'pitch_mean', 'pitch_std', 'pitch_min', 'pit

In [7]:
# ── Cell 4: Load PhoWhisper + Feature Extractor ─────────────────────────────────
from transformers import WhisperModel, WhisperFeatureExtractor
import torch.nn as nn

# ⚠️ FIX: Dùng PhoWhisper-small nếu bộ nhớ hạn chế
PHOWHISPER_MODEL = cfg.get('model', {}).get('backbone', 'vinai/PhoWhisper-base')
if device == 'cpu' and 'large' in PHOWHISPER_MODEL:
    print(f'⚠️  RAM thấp: Dùng vinai/PhoWhisper-small thay vì {PHOWHISPER_MODEL}')
    PHOWHISPER_MODEL = 'vinai/PhoWhisper-small'

feature_extractor = WhisperFeatureExtractor.from_pretrained(PHOWHISPER_MODEL)
print(f'Feature extractor loaded from {PHOWHISPER_MODEL}')


class DialectClassifier(nn.Module):
    """Multi-task PhoWhisper classifier: province + region + ethnicity."""

    def __init__(self, backbone_name, n_province, n_region, n_ethnicity, dropout=0.1):
        super().__init__()
        # Chỉ load model Whisper và sử dụng encoder
        self.whisper = WhisperModel.from_pretrained(backbone_name)
        self.encoder = self.whisper.encoder
        hidden = self.whisper.config.d_model
        
        self.dropout = nn.Dropout(dropout)
        self.province_head  = nn.Linear(hidden, n_province)
        self.region_head    = nn.Linear(hidden, n_region)
        self.ethnicity_head = nn.Linear(hidden, n_ethnicity)

    def forward(self, input_values, attention_mask=None):
        out = self.encoder(input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state.mean(dim=1)  # mean pool trên sequence length
        hidden = self.dropout(hidden)
        return {
            'province' : self.province_head(hidden),
            'region'   : self.region_head(hidden),
            'ethnicity': self.ethnicity_head(hidden),
        }


model = DialectClassifier(
    PHOWHISPER_MODEL,
    n_province=len(province2id),
    n_region=len(region2id),
    n_ethnicity=len(ethnicity2id),
).to(device)

# ⚡ TỐI ƯU 1: FREEZE ENCODER (nhanh hơn 3x, tránh overfit)
for param in model.encoder.parameters():
    param.requires_grad = False
for layer in model.encoder.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')


Feature extractor loaded from vinai/PhoWhisper-base


Loading weights: 100%|██████████| 245/245 [00:00<00:00, 8003.65it/s]
[transformers] WhisperModel LOAD REPORT from: vinai/PhoWhisper-base
Key             | Status     |  | 
----------------+------------+--+-
proj_out.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 58,342,469


In [8]:
# ── Cell 5: Preprocessing (On-the-fly) ──────────────────────────────────────────
MAX_DURATION_S = 8.0  # ⚡ TỐI ƯU: Cắt audio tối đa 8s (RTX 3050)
MAX_SAMPLES = int(MAX_DURATION_S * 16000)  # = 128000 raw samples

import torch
import soundfile as sf
import librosa
import io

def collate_fn(batch):
    audio_arrays = []
    prov_labels = []
    reg_labels = []
    eth_labels = []

    for item in batch:
        path = item['audio']
        if isinstance(path, dict):
            path = path['path']

        # ⚠️ FIX DEADLOCK: Đọc file bằng Python thuần (tránh WSL 9p deadlock)
        with open(path, 'rb') as f:
            audio_bytes = f.read()

        data, sr = sf.read(io.BytesIO(audio_bytes), dtype='float32')

        # ⚠️ FIX ORDER: Phải convert stereo→mono TRƯỚC khi trim
        if data.ndim > 1:
            data = data.mean(axis=1)

        # Resample nếu cần
        if sr != 16000:
            data = librosa.resample(data, orig_sr=sr, target_sr=16000)

        # Cắt khoảng lặng hai đầu
        data, _ = librosa.effects.trim(data, top_db=20)

        # ⚠️ FIX: Truncate raw audio TRƯỚC khi đưa vào feature_extractor
        # Whisper yêu cầu đúng 3000 mel frames — feature_extractor sẽ tự pad
        data = data[:MAX_SAMPLES]

        audio_arrays.append(data)

        prov_labels.append(province2id.get(item['province'], 0))
        reg_labels.append(region2id.get(item['region'], 0))
        eth_labels.append(ethnicity2id.get(item['ethnicity'], 0))

    # ⚠️ FIX: KHÔNG dùng padding='longest' — WhisperFeatureExtractor sẽ tự pad
    # mel spectrogram về đúng 3000 frames (30s). Nếu dùng 'longest' → chỉ 800 frames → ValueError
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors='pt',
        # Để mặc định: tự pad → shape (batch, 80, 3000)
    )

    inputs['province_label']  = torch.tensor(prov_labels, dtype=torch.long)
    inputs['region_label']    = torch.tensor(reg_labels, dtype=torch.long)
    inputs['ethnicity_label'] = torch.tensor(eth_labels, dtype=torch.long)
    return inputs

print('collate_fn ready — Whisper mel shape will be (batch, 80, 3000)')


collate_fn ready — Whisper mel shape will be (batch, 80, 3000)


In [9]:
# ── Cell 6: Training Loop ───────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import mlflow

train_cfg   = cfg.get('training', {})
BATCH_SIZE  = train_cfg.get('batch_size', 4)
if BATCH_SIZE == 'auto': BATCH_SIZE = 4
BATCH_SIZE = int(BATCH_SIZE)
NUM_EPOCHS  = 5 # ⚡ TỐI ƯU 3: Giảm xuống 5 epochs
LR          = train_cfg.get('learning_rate', 1e-4)
GRAD_ACC    = train_cfg.get('gradient_accumulation_steps', 4)
WARMUP      = train_cfg.get('warmup_steps', 100)

# ⚠️ FIX: num_workers=0 (Windows multiprocessing không ổn định với DataLoader)
train_loader = DataLoader(ds['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=False)
val_loader   = DataLoader(ds['valid'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss()

import os
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri(os.path.join(PROJECT_ROOT, 'storage', 'logs', 'mlflow'))
mlflow.set_experiment('dialect-identification')

with mlflow.start_run():
    mlflow.log_params({'model': PHOWHISPER_MODEL, 'batch_size': BATCH_SIZE, 'lr': LR, 'epochs': NUM_EPOCHS})

    scaler = torch.amp.GradScaler('cuda')
    best_val_acc = 0.0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()

        print(f'Start training Epoch {epoch}')
        for step, batch in enumerate(train_loader):
            if step % 10 == 0:
                print(f'  [Epoch {epoch}] Batch {step}/{len(train_loader)}')
            inputs      = batch['input_features'].to(device)
            prov_labels = batch['province_label'].to(device)
            reg_labels  = batch['region_label'].to(device)
            eth_labels  = batch['ethnicity_label'].to(device)

            # ⚡ TỐI ƯU 4: Mixed Precision (nhanh hơn 1.5x)
            with torch.amp.autocast('cuda'):
                logits = model(inputs)
                loss = (
                    criterion(logits['province'],  prov_labels) * 0.6 +
                    criterion(logits['region'],    reg_labels)  * 0.3 +
                    criterion(logits['ethnicity'], eth_labels)  * 0.1
                ) / GRAD_ACC

            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACC

            if (step + 1) % GRAD_ACC == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        correct_prov, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                inputs      = batch['input_features'].to(device)
                prov_labels = batch['province_label'].to(device)
                logits      = model(inputs)
                preds       = logits['province'].argmax(dim=-1)
                correct_prov += (preds == prov_labels).sum().item()
                total        += prov_labels.size(0)

        val_acc = correct_prov / total if total > 0 else 0.0
        mlflow.log_metrics({'train_loss': avg_train_loss, 'val_province_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}: loss={avg_train_loss:.4f}, val_prov_acc={val_acc:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_dialect.pt'))
            print(f'  → New best checkpoint saved (acc={val_acc:.3f})')

    print(f'\nTraining complete. Best val province accuracy: {best_val_acc:.3f}')
    mlflow.log_metric('best_val_province_acc', best_val_acc)


Start training Epoch 0
  [Epoch 0] Batch 0/7204
  [Epoch 0] Batch 10/7204
  [Epoch 0] Batch 20/7204
  [Epoch 0] Batch 30/7204
  [Epoch 0] Batch 40/7204
  [Epoch 0] Batch 50/7204
  [Epoch 0] Batch 60/7204
  [Epoch 0] Batch 70/7204
  [Epoch 0] Batch 80/7204
  [Epoch 0] Batch 90/7204
  [Epoch 0] Batch 100/7204
  [Epoch 0] Batch 110/7204
  [Epoch 0] Batch 120/7204
  [Epoch 0] Batch 130/7204
  [Epoch 0] Batch 140/7204
  [Epoch 0] Batch 150/7204
  [Epoch 0] Batch 160/7204
  [Epoch 0] Batch 170/7204
  [Epoch 0] Batch 180/7204
  [Epoch 0] Batch 190/7204
  [Epoch 0] Batch 200/7204
  [Epoch 0] Batch 210/7204
  [Epoch 0] Batch 220/7204
  [Epoch 0] Batch 230/7204
  [Epoch 0] Batch 240/7204
  [Epoch 0] Batch 250/7204
  [Epoch 0] Batch 260/7204
  [Epoch 0] Batch 270/7204
  [Epoch 0] Batch 280/7204
  [Epoch 0] Batch 290/7204
  [Epoch 0] Batch 300/7204
  [Epoch 0] Batch 310/7204
  [Epoch 0] Batch 320/7204
  [Epoch 0] Batch 330/7204
  [Epoch 0] Batch 340/7204
  [Epoch 0] Batch 350/7204
  [Epoch 0] Batc

In [10]:
# ── Cell 7: Export model ─────────────────────────────────────────────────────────
import json

best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_dialect.pt')
if not os.path.exists(best_ckpt):
    raise RuntimeError(f'Checkpoint không tìm thấy: {best_ckpt}. Hãy chạy Cell 6 trước.')

model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()

os.makedirs(EXPORT_DIR, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'province2id'     : province2id,
    'region2id'       : region2id,
    'ethnicity2id'    : ethnicity2id,
    'backbone'        : PHOWHISPER_MODEL,
    'best_val_acc'    : best_val_acc,
}, os.path.join(EXPORT_DIR, 'dialect_model.pt'))

with open(os.path.join(EXPORT_DIR, 'label_maps.json'), 'w', encoding='utf-8') as f:
    json.dump({'province2id': province2id, 'region2id': region2id, 'ethnicity2id': ethnicity2id},
              f, indent=2, ensure_ascii=False)

print(f'Model exported to {EXPORT_DIR}/')
print(f'Files: {os.listdir(EXPORT_DIR)}')


Model exported to /mnt/d/FPTU/Semester 7/DAT301m/Project/storage/weights/dialect/
Files: ['dialect_model.pt', 'label_maps.json']


## ✓ Dialect Identification Training Complete

**Model saved to**: `storage/weights/dialect/`

**Next**: Run `03_asr_wav2vec2.ipynb`
